# Quantum Circuit Born Machine (QCBM)

This exercise explores Generative AI in Quantum Machine Learning (QML) by implementing a QCBM.

A QCBM is a pure quantum generative model. Unlike classical Generative Adversarial Networks (GANs) that use a neural network to transform noise into data, a QCBM uses a parameterized quantum circuit (PQC) to learn a target probability distribution directly. When we measure the qubits, the outputs are samples drawn from that learned distribution.

We will use Amazon Braket alongside PennyLane (a leading QML library with built-in Braket support) to build, train, and simulate this quantum generative model to learn a simple 2-qubit target distribution.

The GoalTrain a 2-qubit quantum circuit to output the Bell state probability distribution.A Bell state ($|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$) yields the outcomes 00 and 11 with 50% probability each, while 01 and 10 never occur.

## Step 1: Environment Setup
First, make sure you have the required packages installed. If you are running this in an Amazon Braket Notebook instance, these are mostly pre-installed.

In [2]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.0/388.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/15

## Step 2: The Python Code

In [3]:
import pennylane as qml
from pennylane import numpy as np

# 1. Define the Target Distribution (Bell State probabilities)
# The order of states is: 00, 01, 10, 11
target_distribution = np.array([0.5, 0.0, 0.0, 0.5])

# 2. Set up the Amazon Braket Device
# We'll use the local qubit simulator for fast prototyping.
# To use an AWS cloud simulator, change to 'braket.aws.qubit' and provide an s3_destination_folder.
num_wires = 2
dev = qml.device("braket.local.qubit", wires=num_wires)

# 3. Define the Quantum Generator Circuit (QCBM)
@qml.qnode(dev)
def quantum_generator(weights):
    # Layer 1: Arbitrary single-qubit rotations to introduce parameters
    for i in range(num_wires):
        qml.RX(weights[0, i], wires=i)
        qml.RY(weights[1, i], wires=i)

    # Entanglement layer to allow correlation between qubits
    qml.CNOT(wires=[0, 1])

    # Layer 2: Final rotations to shift the probability landscape
    for i in range(num_wires):
        qml.RY(weights[2, i], wires=i)

    # Return the probabilities of all computational basis states (00, 01, 10, 11)
    return qml.probs(wires=range(num_wires))

# 4. Define the Cost Function (Relative Entropy / KL-Divergence)
def kl_divergence(probabilities, target):
    # Add a small epsilon to prevent log(0)
    epsilon = 1e-10
    probabilities = np.clip(probabilities, epsilon, 1.0)
    target = np.clip(target, epsilon, 1.0)
    return np.sum(target * np.log(target / probabilities))

def cost_function(weights):
    generated_probs = quantum_generator(weights)
    return kl_divergence(generated_probs, target_distribution)

# 5. Initialize Parameters and Optimizer
# We need a 3x2 matrix of weights based on our circuit layout
np.random.seed(42)
weights = np.random.uniform(0, 2 * np.pi, (3, num_wires), requires_grad=True)

opt = qml.GradientDescentOptimizer(stepsize=0.1)

# 6. Training Loop
print("Starting QCBM Training...")
print("---------------------------")

for step in range(101):
    weights, cost = opt.step_and_cost(cost_function, weights)

    if step % 20 == 0:
        print(f"Step {step:3d} | Loss (KL-Divergence): {cost:.5f}")

print("---------------------------")
print("Training Complete!\n")

# 7. Evaluate the Results
final_distribution = quantum_generator(weights)
print("Target Distribution:   [00: 0.50,  01: 0.00,  10: 0.00,  11: 0.50]")
print(f"Generated Distribution: [00: {final_distribution[0]:.2f},  01: {final_distribution[1]:.2f},  10: {final_distribution[2]:.2f},  11: {final_distribution[3]:.2f}]")

Starting QCBM Training...
---------------------------
Step   0 | Loss (KL-Divergence): 0.85263
Step  20 | Loss (KL-Divergence): 0.02044
Step  40 | Loss (KL-Divergence): 0.00168
Step  60 | Loss (KL-Divergence): 0.00020
Step  80 | Loss (KL-Divergence): 0.00003
Step 100 | Loss (KL-Divergence): 0.00000
---------------------------
Training Complete!

Target Distribution:   [00: 0.50,  01: 0.00,  10: 0.00,  11: 0.50]
Generated Distribution: [00: 0.50,  01: 0.00,  10: 0.00,  11: 0.50]


## Step 3: Understanding the Mechanics
**The Ansatz:** The structural design of the quantum circuit (quantum_generator) is called an ansatz. It consists of parameterized rotation gates ($RX, RY$) and fixed entangling gates ($CNOT$). By adjusting the rotation angles (weights), we change how the qubits interfere, altering the final measurement probabilities.

**The Loss Metric:** We use Kullback-Leibler (KL) Divergence, a standard classical machine learning loss metric used to measure how much one probability distribution diverges from an expected target distribution.

**The Optimization:** PennyLane automatically calculates the gradients of the quantum circuit parameters using classical backpropagation techniques (or parameter-shift rules on physical QPUs) and passes them to a classical optimizer to update the quantum weights.